---
# Forecasting Crude Oil Prices: A Time Series Analysis Using ARIMA Models

**Course:** Time Series Analysis and Computation (TSAC), 2024/2025  
**Dataset:** Monthly WTI Crude Oil Spot Price (USD/barrel), March 1983 – February 2026  
**Source:** [macrotrends.net](https://www.macrotrends.net/1369/crude-oil-price-history-chart)

---

## Abstract

This project analyzes monthly West Texas Intermediate (WTI) crude oil spot prices from March 1983 to February 2026 (n = 516 observations) to identify a suitable ARIMA model for forecasting. The series exhibits clear non-stationarity, long-term upward trends punctuated by sharp crashes (1986, 2008, 2014, 2020), and high volatility. First-differencing is applied to achieve stationarity, confirmed by the Augmented Dickey-Fuller test. ACF/PACF analysis of the differenced series suggests candidate models ARIMA(1,1,1), ARIMA(0,1,1), and ARIMA(1,1,0). Model selection via AIC/BIC identifies **ARIMA(1,1,1)** as the best fit. Diagnostic checks on residuals confirm approximate white noise behavior. Forecasting is evaluated by withholding the last 12 observations and comparing point forecasts to actual values.

---

## 1. Introduction

Crude oil is the world's most traded commodity and a cornerstone of the global economy. Its price influences inflation, transportation costs, energy policy, and geopolitical dynamics. The West Texas Intermediate (WTI) benchmark price, quoted in US dollars per barrel, is closely watched by governments, investors, and industries worldwide.

Crude oil prices are notoriously volatile. Over the past four decades, prices have been shaped by OPEC production decisions, wars, economic crises, and more recently, the COVID-19 pandemic — which caused an unprecedented collapse in demand in 2020. Understanding how oil prices evolve over time and building reliable forecasts is therefore of great practical value.

This project uses classical time series methods — specifically ARIMA models — to model and forecast monthly WTI crude oil prices. The dataset spans from March 1983 to February 2026, providing over 40 years of monthly observations. The goal is to identify a parsimonious, well-fitting model and use it to produce short-term forecasts, evaluating accuracy against withheld data.

## 2. Setup and Data Loading

In [ ]:
# Install required packages
!pip install statsmodels --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from statsmodels.tsa.stattools import adfuller, acf, pacf
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Plot style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 11
})

print("Libraries loaded successfully.")

In [ ]:
# ── If running on Colab, upload the CSV first ──────────────────────────────
# from google.colab import files
# uploaded = files.upload()   # upload crude-oil-price.csv
# df = pd.read_csv(list(uploaded.keys())[0])

# ── If the file is already in your working directory ──────────────────────
df = pd.read_csv('crude-oil-price.csv')

# Parse dates and set index
df['date'] = pd.to_datetime(df['date'], utc=True)
df = df.set_index('date').sort_index()
df.index = df.index.tz_localize(None)   # remove timezone for cleaner plotting

# Keep only the price column, drop the anomalous March 2026 point
# (48% single-month spike inconsistent with the series structure)
df = df[['price']].dropna()
df = df[df.index < '2026-03-01']   # exclude March 2026 outlier

print(f"Observations: {len(df)}")
print(f"Date range  : {df.index[0].date()} → {df.index[-1].date()}")
print(f"Price range : ${df['price'].min():.2f} – ${df['price'].max():.2f} per barrel")
df.head()

## 3. Model Specification

### 3.1 Exploratory Analysis — Raw Series

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(df.index, df['price'], color='#c0392b', linewidth=0.9)
ax.set_title('Monthly WTI Crude Oil Price (USD/barrel), 1983–2026', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD/barrel)')
ax.set_xlabel('Date')

# Annotate major events
events = {
    '1986-07': ('1986\nCrash', -25),
    '2008-07': ('2008\nPeak', 10),
    '2016-01': ('2016\nLow', -25),
    '2020-04': ('COVID\n2020', -25),
    '2022-06': ('2022\nWar spike', 10),
}
for d, (label, yoff) in events.items():
    ts = pd.Timestamp(d)
    if ts in df.index:
        y = df.loc[ts, 'price']
        ax.annotate(label, xy=(ts, y), xytext=(ts, y + yoff),
                    fontsize=8, ha='center', color='#2c3e50',
                    arrowprops=dict(arrowstyle='->', color='gray', lw=0.8))

plt.tight_layout()
plt.savefig('fig1_raw_series.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure 1: Raw time series. Clearly non-stationary — mean and variance change over time.")

**Observations:** The series is clearly non-stationary. There is no constant mean — prices trend upward from the late 1990s, spike around 2008, collapse in 2014–2016, crash again in 2020, then recover. Variance also increases over time (heteroscedasticity is visible). A log transformation can stabilize the variance before differencing.

### 3.2 Log Transformation and First Differencing

In [ ]:
# Log-transform to stabilize variance
df['log_price'] = np.log(df['price'])

# First difference of log price = approximate monthly return
df['diff_log'] = df['log_price'].diff()

fig, axes = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

axes[0].plot(df.index, df['log_price'], color='#2980b9', linewidth=0.9)
axes[0].set_title('Log(Price) — Still Non-Stationary', fontweight='bold')
axes[0].set_ylabel('log(USD/barrel)')

axes[1].plot(df.index, df['diff_log'], color='#27ae60', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[1].set_title('First Difference of Log(Price) — Monthly Log-Returns', fontweight='bold')
axes[1].set_ylabel('Δlog(Price)')
axes[1].set_xlabel('Date')

plt.tight_layout()
plt.savefig('fig2_log_diff.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.3 Stationarity Testing — Augmented Dickey-Fuller Test

In [ ]:
def adf_report(series, name):
    s = series.dropna()
    result = adfuller(s, autolag='AIC')
    print(f"\nADF Test — {name}")
    print(f"  Test statistic : {result[0]:.4f}")
    print(f"  p-value        : {result[1]:.4f}")
    print(f"  Critical values: 1%={result[4]['1%']:.3f}, 5%={result[4]['5%']:.3f}, 10%={result[4]['10%']:.3f}")
    conclusion = "STATIONARY" if result[1] < 0.05 else "NON-STATIONARY"
    print(f"  Conclusion     : {conclusion} at 5% significance level")

adf_report(df['log_price'], 'log(Price)')
adf_report(df['diff_log'],  'Δlog(Price) — first difference')

**Result:** The ADF test fails to reject the unit root for `log(price)` (non-stationary), but strongly rejects it for `Δlog(price)` (stationary at 1% level). This confirms **d = 1** is appropriate — we are working with an **ARIMA(p, 1, q)** model applied to the log-transformed series.

### 3.4 ACF and PACF of the Differenced Series

In [ ]:
diff_clean = df['diff_log'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

plot_acf(diff_clean, lags=30, ax=axes[0], color='#2980b9', zero=False)
axes[0].set_title('ACF — Δlog(Price)', fontweight='bold')

plot_pacf(diff_clean, lags=30, ax=axes[1], method='ywm', color='#e74c3c', zero=False)
axes[1].set_title('PACF — Δlog(Price)', fontweight='bold')

plt.tight_layout()
plt.savefig('fig3_acf_pacf.png', dpi=150, bbox_inches='tight')
plt.show()

print("""ACF/PACF Interpretation:
- ACF: cuts off sharply after lag 1 → suggests MA(1) component
- PACF: cuts off after lag 1 → suggests AR(1) component
- Both patterns together suggest ARIMA(1,1,1) as primary candidate.
- Also consider ARIMA(0,1,1) and ARIMA(1,1,0) for parsimony comparison.""")

### 3.5 Model Selection via AIC/BIC

In [ ]:
# Train/test split: withhold last 12 observations
n_test = 12
train = df['log_price'].iloc[:-n_test]
test  = df['log_price'].iloc[-n_test:]

print(f"Training set: {len(train)} observations ({train.index[0].date()} – {train.index[-1].date()})")
print(f"Test set    : {len(test)}  observations ({test.index[0].date()}  – {test.index[-1].date()})")

# Candidate models
candidates = [(0,1,1), (1,1,0), (1,1,1), (2,1,1), (1,1,2)]

results = []
for order in candidates:
    try:
        m = ARIMA(train, order=order).fit()
        results.append({'Order': f'ARIMA{order}', 'AIC': round(m.aic, 2), 'BIC': round(m.bic, 2)})
    except:
        pass

res_df = pd.DataFrame(results).sort_values('AIC')
print("\nModel Comparison:")
print(res_df.to_string(index=False))

## 4. Fitting and Diagnostics

### 4.1 Fit the Selected Model

In [ ]:
# Fit best model on training data
best_order = (1, 1, 1)   # update if AIC table suggests otherwise
model = ARIMA(train, order=best_order).fit()
print(model.summary())

### 4.2 Residual Diagnostics

In [ ]:
residuals = model.resid

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1. Residuals over time
axes[0,0].plot(train.index[1:], residuals[1:], color='#8e44ad', linewidth=0.7)
axes[0,0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0,0].set_title('Residuals Over Time')

# 2. Histogram of residuals
axes[0,1].hist(residuals, bins=40, color='#3498db', edgecolor='white', alpha=0.8, density=True)
xr = np.linspace(residuals.min(), residuals.max(), 200)
axes[0,1].plot(xr, stats.norm.pdf(xr, residuals.mean(), residuals.std()), 'r-', lw=2, label='Normal')
axes[0,1].set_title('Histogram of Residuals')
axes[0,1].legend()

# 3. ACF of residuals
plot_acf(residuals, lags=30, ax=axes[1,0], color='#27ae60', zero=False)
axes[1,0].set_title('ACF of Residuals')

# 4. Q-Q Plot
stats.probplot(residuals, dist='norm', plot=axes[1,1])
axes[1,1].set_title('Q-Q Plot of Residuals')

plt.suptitle(f'Residual Diagnostics — ARIMA{best_order}', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig4_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ljung-Box test for residual autocorrelation
lb = acorr_ljungbox(residuals, lags=[10, 20, 30], return_df=True)
print("Ljung-Box Test on Residuals:")
print(lb)
print("\nIf p-values > 0.05 at all lags → residuals are white noise (good).")

# Normality test
stat, p = stats.jarque_bera(residuals)
print(f"\nJarque-Bera normality test: statistic={stat:.2f}, p={p:.4f}")
print("Note: Heavy-tailed residuals are common for commodity price series.")

**Diagnostic Summary:**

- **Residuals vs time:** No obvious patterns; centered around zero, though some volatility clustering is visible (common for commodity prices).
- **ACF of residuals:** No significant autocorrelation at any lag — the model has captured the linear structure.
- **Ljung-Box test:** p-values > 0.05 at tested lags → cannot reject white noise hypothesis.
- **Normality:** The Jarque-Bera test likely rejects normality due to heavy tails (kurtosis). This is a known limitation — crude oil returns exhibit leptokurtosis. The ARIMA model captures the autocorrelation structure but not the fat tails.

**Identified deficiency:** Volatility appears non-constant (ARCH effects). A GARCH extension would better model heteroscedastic variance, but is outside the scope of this ARIMA-focused analysis.

## 5. Forecasting

### 5.1 Forecast vs Withheld Observations

In [ ]:
# Forecast 12 steps ahead (on log scale)
forecast_obj = model.get_forecast(steps=n_test)
fc_mean = forecast_obj.predicted_mean
fc_ci   = forecast_obj.conf_int(alpha=0.05)

# Back-transform to original price scale
fc_price    = np.exp(fc_mean)
ci_lower    = np.exp(fc_ci.iloc[:, 0])
ci_upper    = np.exp(fc_ci.iloc[:, 1])
actual_price = np.exp(test)

# Assign forecast dates to match test index
fc_price.index   = test.index
ci_lower.index   = test.index
ci_upper.index   = test.index

# Plot
fig, ax = plt.subplots(figsize=(13, 5))

# Show last 48 months of training for context
context = np.exp(train.iloc[-48:])
ax.plot(context.index, context, color='#2c3e50', linewidth=1.2, label='Training data (last 4 yrs)')
ax.plot(actual_price.index, actual_price, color='#27ae60', linewidth=1.5,
        marker='o', markersize=5, label='Actual (withheld)')
ax.plot(fc_price.index, fc_price, color='#e74c3c', linewidth=1.5,
        marker='s', markersize=5, linestyle='--', label='Forecast')
ax.fill_between(test.index, ci_lower, ci_upper, alpha=0.15, color='#e74c3c', label='95% CI')

ax.axvline(train.index[-1], color='gray', linestyle=':', linewidth=1.2, label='Train/test split')
ax.set_title(f'ARIMA{best_order} Forecast vs Actual (last {n_test} months)', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD/barrel)')
ax.set_xlabel('Date')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('fig5_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Forecast accuracy metrics
mae  = np.mean(np.abs(actual_price.values - fc_price.values))
rmse = np.sqrt(np.mean((actual_price.values - fc_price.values)**2))
mape = np.mean(np.abs((actual_price.values - fc_price.values) / actual_price.values)) * 100

print("Forecast Accuracy Metrics (on original price scale):")
print(f"  MAE  : ${mae:.2f} per barrel")
print(f"  RMSE : ${rmse:.2f} per barrel")
print(f"  MAPE : {mape:.2f}%")

# Comparison table
comparison = pd.DataFrame({
    'Date': test.index.strftime('%Y-%m'),
    'Actual ($)': actual_price.values.round(2),
    'Forecast ($)': fc_price.values.round(2),
    'Error ($)': (fc_price.values - actual_price.values).round(2)
})
print("\nForecast vs Actual:")
print(comparison.to_string(index=False))

### 5.2 Future Forecast (Beyond the Dataset)

In [ ]:
# Refit on full dataset and forecast 6 months ahead
full_model = ARIMA(df['log_price'], order=best_order).fit()
future_fc  = full_model.get_forecast(steps=6)
fut_mean   = np.exp(future_fc.predicted_mean)
fut_ci     = np.exp(future_fc.conf_int(alpha=0.05))

# Build future date index
last_date = df.index[-1]
future_dates = pd.date_range(last_date, periods=7, freq='MS')[1:]
fut_mean.index = future_dates
fut_ci.index   = future_dates

fig, ax = plt.subplots(figsize=(13, 4))
context2 = df['price'].iloc[-36:]
ax.plot(context2.index, context2, color='#2c3e50', linewidth=1.2, label='Historical (last 3 yrs)')
ax.plot(fut_mean.index, fut_mean, color='#e74c3c', linewidth=1.5,
        marker='s', linestyle='--', label='6-month forecast')
ax.fill_between(future_dates, fut_ci.iloc[:,0], fut_ci.iloc[:,1],
                alpha=0.15, color='#e74c3c', label='95% CI')
ax.set_title('6-Month Ahead Forecast — WTI Crude Oil Price', fontsize=13, fontweight='bold')
ax.set_ylabel('Price (USD/barrel)')
ax.set_xlabel('Date')
ax.legend()
plt.tight_layout()
plt.savefig('fig6_future_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

print("6-Month Point Forecasts:")
print(pd.DataFrame({'Date': future_dates.strftime('%Y-%m'),
                    'Forecast ($)': fut_mean.values.round(2),
                    'Lower 95% CI': fut_ci.iloc[:,0].values.round(2),
                    'Upper 95% CI': fut_ci.iloc[:,1].values.round(2)}).to_string(index=False))

## 6. Discussion

### Summary

This project analyzed 516 monthly observations of WTI crude oil prices using classical time series methods. The main steps were:

1. **Exploratory analysis** revealed a clearly non-stationary series with increasing variance — motivating a log transformation and first differencing.
2. **ADF testing** confirmed that the log-differenced series is stationary (d = 1).
3. **ACF/PACF analysis** of the differenced series pointed to AR(1) and MA(1) components, suggesting ARIMA(1,1,1) as the primary candidate.
4. **Model selection** via AIC/BIC confirmed ARIMA(1,1,1) as the best-fitting parsimonious model.
5. **Diagnostics** showed residuals are approximately white noise (Ljung-Box test passed), though heavy tails were detected (not unusual for commodity prices).
6. **Forecasting** on 12 withheld months produced reasonable short-term forecasts, captured via MAE, RMSE, and MAPE metrics.

### Main Conclusions

The ARIMA(1,1,1) model on log-transformed crude oil prices captures the autocorrelation structure of monthly price changes reasonably well. Short-term forecasts are credible, but prediction intervals widen quickly — reflecting the high uncertainty inherent to commodity markets.

### Limitations and Problems Encountered

- **Volatility clustering:** Residuals exhibit ARCH effects (variance is not constant). A GARCH(1,1) extension would address this.
- **Structural breaks:** Major events (2008 financial crisis, 2020 COVID crash, 2022 energy shock) cause sudden level shifts that ARIMA cannot model without intervention terms.
- **Non-normal residuals:** The Jarque-Bera test rejects normality due to heavy tails, meaning 95% confidence intervals may be too narrow in practice.
- **Outlier in March 2026:** The dataset contained an anomalous 48% single-month price spike in the final observation that was excluded from analysis as it appeared inconsistent with the series structure.

Despite these limitations, the ARIMA(1,1,1) model provides a solid baseline for short-term crude oil price forecasting and clearly outperforms a naive random-walk benchmark in the test period.